# AE2 – Manejo de archivos CSV, JSON y TXT

**Analítica de Datos – IU Digital de Antioquia** · PREICA2602B020125 · Kevin Daniel Sepúlveda Giraldo

En este notebook coordino tres clases que construí, cada una en su propio archivo `.py`:

| Clase | Archivo | Qué hace |
|---|---|---|
| `ManejadorCSV` | `manejador_csv.py` | Importa con `csv.reader`, modifica y guarda un CSV nuevo con `csv.writer` |
| `ManejadorJSON` | `manejador_json.py` | Importa con `json.load`, modifica y guarda un JSON nuevo con `json.dump` |
| `GeneradorResumenTXT` | `generador_resumen_txt.py` | Recibe los resultados de las otras dos clases y genera `resumen.txt` |

**Flujo:** descarga desde Hugging Face → importar → modificar → guardar nuevo CSV y JSON → tercera clase → `resumen.txt` con URL de origen

## 1. URL de origen y rutas de los archivos

Descargué los dos archivos desde Hugging Face y los guardé sin cambios en `datos/originales/`.
Dejo las URL en una variable para enviarlas después al resumen.

In [1]:
from pathlib import Path

URLS_ORIGEN = {
    "CSV": "https://huggingface.co/datasets/aptologia/barrios-mas-baratos-para-arrendar-colombia-2026/blob/main/data/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv",
    "JSON": "https://huggingface.co/datasets/Vault-of-History/colombian-numismatic-collection/blob/main/colombian-coins.json",
}

CSV_ORIGINAL = Path("datos/originales/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv")
JSON_ORIGINAL = Path("datos/originales/colombian-coins.json")

CSV_MODIFICADO = Path("datos/modificados/barrios_arriendo_modificado.csv")
JSON_MODIFICADO = Path("datos/modificados/monedas_colombia_modificado.json")

for ruta in (CSV_ORIGINAL, JSON_ORIGINAL):
    print(f"{ruta} -> existe: {ruta.exists()}")

datos/originales/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv -> existe: True
datos/originales/colombian-coins.json -> existe: True


## 2. Importar las clases e inicializar los objetos

In [2]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON
from generador_resumen_txt import GeneradorResumenTXT

manejador_csv = ManejadorCSV(CSV_ORIGINAL, URLS_ORIGEN["CSV"])
manejador_json = ManejadorJSON(JSON_ORIGINAL, URLS_ORIGEN["JSON"])

## 3. Importar los archivos locales

Cada clase abre su archivo con `with open(...)` y lo lee con su módulo (`csv.reader` y `json.load`).

In [3]:
filas_csv = manejador_csv.importar()
registros_json = manejador_json.importar()

print(f"CSV importado:  {filas_csv} filas | encabezados: {manejador_csv.encabezados}")
print(f"JSON importado: {registros_json} registros")

CSV importado:  145 filas | encabezados: ['city', 'barrio', 'avisos', 'canon_cop_mediana']
JSON importado: 14 registros


## 4. Modificar el CSV (barrios más baratos para arrendar en Colombia)

**Antes:**

In [4]:
for fila in manejador_csv.vista_previa(5):
    print(fila)

{'city': 'Bogotá', 'barrio': 'Madelena', 'avisos': '30', 'canon_cop_mediana': '1300000.0'}
{'city': 'Jamundí', 'barrio': 'Parque Natura', 'avisos': '33', 'canon_cop_mediana': '1300000.0'}
{'city': 'Barranquilla', 'barrio': 'Alameda del Río', 'avisos': '156', 'canon_cop_mediana': '1317000.0'}
{'city': 'El Carmen de Viboral', 'barrio': 'El Carmen de Viboral', 'avisos': '39', 'canon_cop_mediana': '1400000.0'}
{'city': 'Bogotá', 'barrio': 'Fontibon', 'avisos': '81', 'canon_cop_mediana': '1450000.0'}


Hago tres modificaciones con métodos de la clase:

1. **Agregar** la columna `canon_millones` (canon mediano en millones de pesos), para leerlo más fácil.
2. **Actualizar** el valor `Bogotá` por `Bogotá D.C.` en la columna `city`.
3. **Eliminar** los barrios con menos de 35 avisos, porque con pocos anuncios la mediana del canon es menos confiable.

In [5]:
manejador_csv.agregar_columna(
    "canon_millones",
    lambda fila: round(float(fila["canon_cop_mediana"]) / 1_000_000, 2),
)
cambiados = manejador_csv.actualizar_valor("city", "Bogotá", "Bogotá D.C.")
eliminados = manejador_csv.eliminar_filas(
    lambda fila: int(fila["avisos"]) < 35,
    "barrios con menos de 35 avisos",
)

print(f"Filas con 'Bogotá' actualizadas: {cambiados}")
print(f"Filas eliminadas: {eliminados}")
print(f"Filas antes: {filas_csv} | Filas después: {len(manejador_csv.filas)}")

Filas con 'Bogotá' actualizadas: 34
Filas eliminadas: 20
Filas antes: 145 | Filas después: 125


**Después:**

In [6]:
for fila in manejador_csv.vista_previa(5):
    print(fila)

{'city': 'Barranquilla', 'barrio': 'Alameda del Río', 'avisos': '156', 'canon_cop_mediana': '1317000.0', 'canon_millones': '1.32'}
{'city': 'El Carmen de Viboral', 'barrio': 'El Carmen de Viboral', 'avisos': '39', 'canon_cop_mediana': '1400000.0', 'canon_millones': '1.4'}
{'city': 'Bogotá D.C.', 'barrio': 'Fontibon', 'avisos': '81', 'canon_cop_mediana': '1450000.0', 'canon_millones': '1.45'}
{'city': 'Bogotá D.C.', 'barrio': 'Castilla', 'avisos': '35', 'canon_cop_mediana': '1450600.0', 'canon_millones': '1.45'}
{'city': 'Medellín', 'barrio': 'San Antonio de Prado', 'avisos': '53', 'canon_cop_mediana': '1500000.0', 'canon_millones': '1.5'}


## 5. Modificar el JSON (colección de monedas colombianas)

**Antes** (solo algunos campos para que se vea claro):

In [7]:
CAMPOS_A_MOSTRAR = ["_id", "year_of_issue", "denomination", "unit", "status", "photo"]

for registro in manejador_json.vista_previa(3, CAMPOS_A_MOSTRAR):
    print(registro)

{'_id': 'CO-1977-10C-PORTRAIT', 'year_of_issue': 1977, 'denomination': 10, 'unit': 'centavos', 'status': 'No longer in circulation', 'photo': 'IMG_20260812_114751866.jpg'}
{'_id': 'CO-1971-50C-PORTRAIT', 'year_of_issue': 1971, 'denomination': 50, 'unit': 'centavos', 'status': 'No longer in circulation', 'photo': 'IMG_20260812_114804319.jpg'}
{'_id': 'CO-1959-50C-ARMS', 'year_of_issue': 1959, 'denomination': 50, 'unit': 'centavos', 'status': 'No longer in circulation', 'photo': 'IMG_20260812_115026084.jpg'}


Hago tres modificaciones con métodos de la clase:

1. **Agregar** el campo `antiguedad_anios` (años desde que se emitió la moneda hasta 2026).
2. **Actualizar** el estado `No longer in circulation` por `Fuera de circulación`.
3. **Eliminar** el campo `photo`, que es solo el nombre de una foto que no viene en el dataset.

In [8]:
manejador_json.agregar_campo(
    "antiguedad_anios",
    lambda moneda: 2026 - moneda["year_of_issue"] if isinstance(moneda.get("year_of_issue"), int) else None,
)
actualizados = manejador_json.actualizar_valor("status", "No longer in circulation", "Fuera de circulación")
sin_foto = manejador_json.eliminar_campo("photo")

print(f"Registros con estado actualizado: {actualizados}")
print(f"Registros a los que se les quitó 'photo': {sin_foto}")

Registros con estado actualizado: 14
Registros a los que se les quitó 'photo': 14


**Después:**

In [9]:
CAMPOS_DESPUES = ["_id", "year_of_issue", "antiguedad_anios", "denomination", "unit", "status"]

for registro in manejador_json.vista_previa(3, CAMPOS_DESPUES):
    print(registro)

quedan_con_foto = sum("photo" in moneda for moneda in manejador_json.registros)
print(f"\nRegistros que todavía tienen el campo 'photo': {quedan_con_foto}")

{'_id': 'CO-1977-10C-PORTRAIT', 'year_of_issue': 1977, 'antiguedad_anios': 49, 'denomination': 10, 'unit': 'centavos', 'status': 'Fuera de circulación'}
{'_id': 'CO-1971-50C-PORTRAIT', 'year_of_issue': 1971, 'antiguedad_anios': 55, 'denomination': 50, 'unit': 'centavos', 'status': 'Fuera de circulación'}
{'_id': 'CO-1959-50C-ARMS', 'year_of_issue': 1959, 'antiguedad_anios': 67, 'denomination': 50, 'unit': 'centavos', 'status': 'Fuera de circulación'}

Registros que todavía tienen el campo 'photo': 0


## 6. Guardar los archivos nuevos y verificar los cambios

Guardo los archivos con nombres nuevos en `datos/modificados/`, así los originales quedan intactos.
Para verificar, vuelvo a leer los archivos guardados con objetos nuevos de las mismas clases.

In [10]:
ruta_csv = manejador_csv.guardar(CSV_MODIFICADO)
ruta_json = manejador_json.guardar(JSON_MODIFICADO)
print(f"CSV guardado en:  {ruta_csv}")
print(f"JSON guardado en: {ruta_json}")

verificacion_csv = ManejadorCSV(ruta_csv, URLS_ORIGEN["CSV"])
verificacion_json = ManejadorJSON(ruta_json, URLS_ORIGEN["JSON"])

print(f"\nCSV nuevo:  {verificacion_csv.importar()} filas | encabezados: {verificacion_csv.encabezados}")
print(f"¿Tiene la columna canon_millones? {'canon_millones' in verificacion_csv.encabezados}")

print(f"\nJSON nuevo: {verificacion_json.importar()} registros")
primera = verificacion_json.registros[0]
print(f"¿Tiene antiguedad_anios? {'antiguedad_anios' in primera} | "
      f"¿Todavía tiene photo? {'photo' in primera} | status: {primera.get('status')}")

print(f"\nLos originales siguen ahí: {CSV_ORIGINAL.exists()} y {JSON_ORIGINAL.exists()}")

CSV guardado en:  datos/modificados/barrios_arriendo_modificado.csv
JSON guardado en: datos/modificados/monedas_colombia_modificado.json

CSV nuevo:  125 filas | encabezados: ['city', 'barrio', 'avisos', 'canon_cop_mediana', 'canon_millones']
¿Tiene la columna canon_millones? True

JSON nuevo: 14 registros
¿Tiene antiguedad_anios? True | ¿Todavía tiene photo? False | status: Fuera de circulación

Los originales siguen ahí: True y True


## 7. Generar el resumen con la tercera clase

La clase `GeneradorResumenTXT` recibe las mismas URL de origen de este notebook y el informe de cada clase
(archivos, modificaciones, métodos usados y cantidad de registros), y escribe `resumen.txt`.

In [11]:
generador = GeneradorResumenTXT(
    titulo="Resumen AE2 - Manejo de archivos CSV y JSON",
    urls_origen=URLS_ORIGEN,
)
generador.agregar_informe(manejador_csv.informe())
generador.agregar_informe(manejador_json.informe())

ruta_resumen = generador.guardar("resumen.txt")
print(f"Resumen guardado en: {ruta_resumen}\n")
print(ruta_resumen.read_text(encoding="utf-8"))

Resumen guardado en: resumen.txt

Resumen AE2 - Manejo de archivos CSV y JSON
Generado: 2026-10-26 12:28

URL DE ORIGEN
- CSV: https://huggingface.co/datasets/aptologia/barrios-mas-baratos-para-arrendar-colombia-2026/blob/main/data/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv
- JSON: https://huggingface.co/datasets/Vault-of-History/colombian-numismatic-collection/blob/main/colombian-coins.json

ARCHIVO CSV - clase ManejadorCSV
----------------------------------------------------------------------
URL de origen:        https://huggingface.co/datasets/aptologia/barrios-mas-baratos-para-arrendar-colombia-2026/blob/main/data/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv
Archivo importado:    datos/originales/barrios_mas_baratos_para_arrendar_colombia_2026_datos.csv
Archivo guardado:     datos/modificados/barrios_arriendo_modificado.csv
Registros importados: 145
Registros guardados:  125
Métodos utilizados:   importar, agregar_columna, actualizar_valor, eliminar_fil